In [3]:
import math
import hashlib
from collections import Counter
#from langchain.docstore.document import Document
from langchain_core.documents import Document 

# ============================================================
# Simple TF-IDF Based Document Encoder (No Third-Party Embeddings)
# ============================================================

class SimpleDocumentEncoder:
    def __init__(self):
        self.vocabulary = {}
        self.idf_scores = {}
        self.vocab_size = 0

    def _tokenize(self, text: str) -> list[str]:
        """Simple whitespace + punctuation tokenizer."""
        text = text.lower()
        # Remove common punctuation
        for char in ".,!?;:\"'()-[]{}":
            text = text.replace(char, " ")
        return [token for token in text.split() if token.strip()]

    def _build_vocabulary(self, documents: list[Document]):
        """Build vocabulary from a list of LangChain Documents."""
        for doc in documents:
            tokens = self._tokenize(doc.page_content)
            for token in tokens:
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)
        self.vocab_size = len(self.vocabulary)

    def _compute_tf(self, tokens: list[str]) -> dict[str, float]:
        """Compute Term Frequency for a given list of tokens."""
        counter = Counter(tokens)
        total_tokens = len(tokens)
        return {token: count / total_tokens for token, count in counter.items()}

    def _compute_idf(self, documents: list[Document]):
        """Compute Inverse Document Frequency across all documents."""
        total_docs = len(documents)
        doc_freq = Counter()
        for doc in documents:
            unique_tokens = set(self._tokenize(doc.page_content))
            for token in unique_tokens:
                doc_freq[token] += 1
        self.idf_scores = {
            token: math.log(total_docs / (1 + freq))
            for token, freq in doc_freq.items()
        }

    def _compute_tfidf_vector(self, tokens: list[str]) -> list[float]:
        """Convert tokens into a TF-IDF vector based on built vocabulary."""
        tf = self._compute_tf(tokens)
        vector = [0.0] * self.vocab_size
        for token, tf_score in tf.items():
            if token in self.vocabulary:
                idx = self.vocabulary[token]
                idf_score = self.idf_scores.get(token, 0.0)
                vector[idx] = tf_score * idf_score
        return vector

    def _normalize(self, vector: list[float]) -> list[float]:
        """L2 normalize a vector."""
        magnitude = math.sqrt(sum(v ** 2 for v in vector))
        if magnitude == 0:
            return vector
        return [v / magnitude for v in vector]

    def fit(self, documents: list[Document]):
        """Fit the encoder on a list of LangChain Documents."""
        self._build_vocabulary(documents)
        self._compute_idf(documents)
        print(f"Vocabulary size: {self.vocab_size}")

    def encode(self, documents: list[Document]) -> list[list[float]]:
        """Encode a list of LangChain Documents into TF-IDF vectors."""
        embeddings = []
        for doc in documents:
            tokens = self._tokenize(doc.page_content)
            vector = self._compute_tfidf_vector(tokens)
            normalized_vector = self._normalize(vector)
            embeddings.append(normalized_vector)
        return embeddings

    def encode_single(self, document: Document) -> list[float]:
        """Encode a single LangChain Document into a TF-IDF vector."""
        tokens = self._tokenize(document.page_content)
        vector = self._compute_tfidf_vector(tokens)
        return self._normalize(vector)


# ============================================================
# Utility: Cosine Similarity
# ============================================================

def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    mag_a = math.sqrt(sum(a ** 2 for a in vec_a))
    mag_b = math.sqrt(sum(b ** 2 for b in vec_b))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot_product / (mag_a * mag_b)


# ============================================================
# Demo
# ============================================================

if __name__ == "__main__":
    # Sample LangChain Documents
    documents = [
        Document(page_content="Python is a popular programming language for web development.", metadata={"source": "doc1"}),
        Document(page_content="Flask is a lightweight web framework built with Python.", metadata={"source": "doc2"}),
        Document(page_content="Machine learning models require large datasets for training.", metadata={"source": "doc3"}),
        Document(page_content="Deep learning is a subset of machine learning using neural networks.", metadata={"source": "doc4"}),
        Document(page_content="Web development involves HTML, CSS, and JavaScript.", metadata={"source": "doc5"}),
    ]

    # Initialize and fit encoder
    encoder = SimpleDocumentEncoder()
    encoder.fit(documents)

    # Encode all documents
    embeddings = encoder.encode(documents)

    # Print embedding dimensions
    print(f"\nTotal documents encoded: {len(embeddings)}")
    print(f"Embedding dimension: {len(embeddings[0])}")

    # Similarity check
    print("\n--- Similarity Check ---")
    query = Document(page_content="Python web framework development")
    query_vector = encoder.encode_single(query)

    print(f"\nQuery: '{query.page_content}'\n")
    for i, doc in enumerate(documents):
        similarity = cosine_similarity(query_vector, embeddings[i])
        print(f"  Doc {i+1} | Similarity: {similarity:.4f} | {doc.page_content[:60]}...")

Vocabulary size: 32

Total documents encoded: 5
Embedding dimension: 32

--- Similarity Check ---

Query: 'Python web framework development'

  Doc 1 | Similarity: 0.2590 | Python is a popular programming language for web development...
  Doc 2 | Similarity: 0.4511 | Flask is a lightweight web framework built with Python....
  Doc 3 | Similarity: 0.0000 | Machine learning models require large datasets for training....
  Doc 4 | Similarity: 0.0000 | Deep learning is a subset of machine learning using neural n...
  Doc 5 | Similarity: 0.1232 | Web development involves HTML, CSS, and JavaScript....
